# XAI in Sports Analytics

## Table of Contents

1. Overview
2. Q&As
3. Dependencies
4. Data
5. Model Training
6. Model Evaluation
7. Model Interpretation
8. Summary
9. Exercises

## 1. Overview

Injuries are a significant concern in professional sports, affecting both player health and team performance. Imagine if we could use machine learning to predict which athletes might be at higher risk of injury. This could revolutionize training regimens, player management, and even transfer strategies.

Consider this statement from a leading sports scientist:

> By 2025, we predict that 80% of professional sports teams will be using AI-driven injury prediction models, potentially reducing season-ending injuries by up to 25%.

With this in mind, let's explore how we can leverage machine learning to identify athletes at risk of injury. Remember, a simple "You're likely to get injured!" won't cut it. Context and clear explanations will be crucial for players, coaches, and medical staff to make informed decisions.

Now, let's look at some questions our athletes and coaches may want answered.

## 2. Q&As

When an AI system flags an athlete as high-risk for injury, they'd likely ask:

1. Why am I considered at high risk for injury?
2. What factors contributed to this prediction?
3. What should I do to reduce my risk?

Potential answers might sound like:

1. Your recent increase in training intensity, combined with your age and injury history, has significantly elevated your risk profile.
2. The model considered factors such as your recent performance metrics, sleep patterns, and biomechanical data from your last game. The combination of these factors, especially your decreased recovery time between high-intensity sessions, raised red flags.
3. We recommend adjusting your training schedule to include more rest days, focusing on specific strength exercises, and closely monitoring your sleep quality. We'll also conduct more frequent physical assessments to track your progress.

This kind of detailed explanation would help athletes and staff feel more confident about the AI's recommendations. Now, let's dive into our example.

## 3. Dependencies

Here are the packages we will be using in this notebook.

- `scikit-learn`
- `pandas`
- `joblib`
- `matplotlib`
- `alibi`
- `statsmodels`
- `mlserver`

In [ ]:
!pip install scikit-learn pandas joblib matplotlib alibi numpy rich mlserver

## 4. Data

We'll be using a dataset of professional soccer players. This dataset contains information such as age, position, physical metrics, and injury history. While it's a simplified version of what a real sports analytics team might use, it serves as an excellent starting point for our injury prediction model.

Why does this matter? Machine learning excels at finding patterns in data, and we should leverage these tools responsibly to enhance athlete performance and longevity. Many career-ending injuries could potentially be prevented with early detection and intervention. If we can help athletes of all levels while respecting their privacy and rights, we should push the boundaries of what's possible in sports science.

Here's a brief description of our variables:
- `Age` - player's age
- `Position` - playing position (Forward, Midfielder, Defender, Goalkeeper)
- `Games_Played` - number of games played this season
- `Minutes_Played` - total minutes played this season
- `Sprint_Distance` - total sprint distance in kilometers
- `High_Intensity_Distance` - total high-intensity running distance in kilometers
- `Previous_Injuries` - number of previous injuries
- `Sleep_Quality` - average sleep quality score (1-10)
- `Recovery_Time` - average recovery time between games in hours
- `Injury_Risk` - target variable (0: Low risk, 1: High risk)

Let's start by loading and evaluating our data.

In [ ]:
from sklearn.model_selection import train_test_split
from rich import print
import pandas as pd
import numpy as np

# Generate synthetic data
np.random.seed(42)
n_samples = 1000

data = {
    'Age': np.random.randint(18, 35, n_samples),
    'Position': np.random.choice(['Forward', 'Midfielder', 'Defender', 'Goalkeeper'], n_samples),
    'Games_Played': np.random.randint(0, 50, n_samples),
    'Minutes_Played': np.random.randint(0, 4500, n_samples),
    'Sprint_Distance': np.random.uniform(0, 100, n_samples),
    'High_Intensity_Distance': np.random.uniform(0, 200, n_samples),
    'Previous_Injuries': np.random.randint(0, 5, n_samples),
    'Sleep_Quality': np.random.uniform(5, 10, n_samples),
    'Recovery_Time': np.random.uniform(24, 120, n_samples),
    'Injury_Risk': np.random.randint(0, 2, n_samples)
}

df = pd.DataFrame(data)
df.head()

In [ ]:
df.shape

In [ ]:
y = df['Injury_Risk']
X = df.drop(['Injury_Risk'], axis=1).copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=9)

## 5. Model Training

For this section, we'll use logistic regression due to its high interpretability and ease of use.

If you're new to logistic regression, think of it as a classification algorithm used to predict a binary outcome (e.g., high injury risk or low injury risk).

Here's a sports-specific example:

Suppose you're a team's data analyst trying to predict if a player will get injured in the next match (a binary yes/no outcome) based on their recent performance metrics, training load, and recovery time.

1. Convert the output to a probability between 0-1, representing the chance of injury.

2. Use a linear model to combine input features and calculate a 'score':
   
   $score = Intercept + Age * \beta_1 + Minutes_Played * \beta_2 + Sprint_Distance * \beta_3 + ...$

3. Convert this score to a probability using the logistic function:
   
   $probability = \frac{1}{(1 + e^{(-score)})}$

4. If probability > 0.5, predict high injury risk. Otherwise, predict low risk.

This simplified explanation should give you an intuition of how the method works. Now, let's train our model.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set(rc={'figure.figsize':(11.7,8.27)})

Feel free to experiment with the parameters below.

In [ ]:
lr_cls = LogisticRegression(random_state=0, max_iter=500, verbose=0)

In [ ]:
lr_cls.fit(X_train, y_train)

In [ ]:
lr_cls.coef_

If you don't have the path below, you can create with the following command in the termianl.

```sh
mkdir -p models/diabetes/
```

In [ ]:
model_path = 'models/diabetes/lr_cls_diabetes.pkl'

In [ ]:
joblib.dump(lr_cls, model_path)

In [ ]:
!ls models/diabetes

In [ ]:
lr_cls = joblib.load(model_path)

Let's do a quick sanity check before we move on to thoroughly evaluating our model. For this, we will 
pick a random sample from the test dataset.

In [ ]:
x = X_test.sample(1)
y = y_test[x.index[0]]
print(f"Actual Injury Risk: {'High' if y == 1 else 'Low'}")
x

In [ ]:
lr_cls.classes_

In [ ]:
probabilities = lr_cls.predict_proba(x)
print(f"Probability of Low Injury Risk: {probabilities[0][0]:.2f}")
print(f"Probability of High Injury Risk: {probabilities[0][1]:.2f}")
print(f"Predicted Injury Risk: {'High' if probabilities[0][1] > 0.5 else 'Low'}")

In [ ]:
y_pred = lr_cls.predict(X_test)
accuracy = (y_pred == y_test).mean()
print(f'Model Accuracy: {accuracy:.2f}')

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

title = 'Confusion matrix for Injury Risk Prediction'
disp = ConfusionMatrixDisplay.from_estimator(
    lr_cls, X_test, y_test, 
    display_labels=['Low Risk', 'High Risk'],
    cmap=plt.cm.Blues, normalize=None
)
disp.ax_.set_title(title);

## 6. Model Evaluation

The first method we'll explore is called [Kernel SHAP](https://shap-lrjball.readthedocs.io/en/latest/generated/shap.KernelExplainer.html).

> It's a model interpretation method that explains individual predictions by determining the contribution of each feature. It uses concepts from game theory and local surrogate models to quantify feature importance.

Here's a sports analogy to understand Kernel SHAP:

Imagine a team performance prediction model. The inputs are player stats like goals scored, assists, tackles, and minutes played. The model predicts how well the team will perform.

To explain an individual prediction, Kernel SHAP is like asking:

"How much did each player stat contribute to the overall team performance prediction?" 

It determines the SHAP value, or impact, of each feature by comparing team performances with and without that stat. Goals scored might get a high positive SHAP value because they significantly improve predicted performance. Minutes played might have a lower SHAP value since it has less direct impact. By summing the SHAP values for all features, Kernel SHAP explains the total predicted performance.

This way, Kernel SHAP attributes the prediction of any complex model to each input feature, making model behavior more interpretable.

Let's implement `KernelShap` for our injury risk model.

In [ ]:
from alibi.explainers import KernelShap

In [ ]:
explainer = KernelShap(lr_cls.predict_proba, task='classification')
explainer

Explainers in Alibi work in the same fashion as estimators in sklearn, that is, they follow the 
`.fit()` and `.predict()` way of doing things so if you are familiar with sklearn, this step will 
feel familiar to you.

In [ ]:
explainer.fit(X_train)

Once we finish creating an explainer, the object we get back gives us a lot of useful information like the one above.

Note that, running an explainer in a large batch of data can be quite compute intensive (depending on the 
explainer of course), so it is good practice to save your models once your code finishes creating them. Let's 
save ours, load it and test it again.

In [ ]:
explainer_path = 'models/diabetes/lr_cls_explainer.pkl'

In [ ]:
joblib.dump(explainer, explainer_path)

In [ ]:
explainer = joblib.load(explainer_path)

In [ ]:
x = X_test.sample(1)
y = y_test[x.index].iloc[0]
print(y)
x

In [ ]:
features = X_train.columns.to_list()
features

As you might have noticed in the metadata returned when we trained our model, `KernelShap` is both local and global. This means it can be applied to one or many samples at a time. Let's try it on our random sample from earlier.

In [ ]:
result = explainer.explain(x)

In [ ]:
result.shap_values[0]

In [ ]:
explainer.predictor(x)

What we're interested in is the `shap_values` returned by our explainer. Let's see what these look like.

In [ ]:
result.shap_values

In [ ]:
def plot_importance(feat_imp, feat_names, class_idx):
    df = pd.DataFrame(data=feat_imp, columns=feat_names).sort_values(by=0, axis='columns')
    feat_imp, feat_names = df.values[0], df.columns
    fig, ax = plt.subplots(figsize=(10, 5))
    y_pos = np.arange(len(feat_imp))
    ax.barh(y_pos, feat_imp)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feat_names, fontsize=15)
    ax.invert_yaxis()
    ax.set_xlabel(f'Feature effects for class {class_idx}', fontsize=15)
    return ax, fig

In [ ]:
import numpy as np

In [ ]:
plot_importance(result.shap_values[1], features, 'High Injury Risk');

In [ ]:
import shap

In [ ]:
result = explainer.explain(X_train[:100])

In [ ]:
shap.summary_plot(result.shap_values[0], X_train[:100], features);

A positive SHAP value means the feature pushed the output higher. Negative means it pushed the output lower.

It is important to note that, if we train the explainer on a large amount of data (with some compute expenses), the 
explainer would have learned enough about the model globally to locally explain the interactions for new cases.

## 7. Model Interpretation

While Kernel Shap is considered a black-box method for explaining our model, because we chose logistic regression as our classifier, we can also interrogate each of the model's coefficients for further interpretation.

We'll use `statsmodels` to fit the model again because it provides a nice summary table.

In [ ]:
# !pip install 'alibi[shap]'
!pip install 'alibi[ray]'

In [ ]:
import statsmodels.api as sm

In [ ]:
log_model = sm.Logit(y_train, sm.add_constant(X_train))
log_result = log_model.fit()

In [ ]:
print(log_result.summary2())

In the table above we can examine not only the coefficients of each parameter, but also the standard deviation and 
AIC and BIC values of our model.

Because the coefficients are the logarithms of the odds (i.e. the probability of a positive case over 
the probability of a negative case), we can convert them back into exponentials to get a better sense of 
what each value means.

In [ ]:
np.exp(log_result.params).sort_values(ascending=False)

What do these odds mean for a player's injury risk? It means that the odds of being at high risk for injury increase by a factor shown for each additional unit of the respective feature, provided every other feature stays unchanged.

For instance, if 'Previous_Injuries' has an odds ratio of 1.5, it means that for each additional previous injury, the odds of being at high risk increase by 50%, assuming all other factors remain constant.

We need more context for this, which we can get by considering the standard deviation of each feature.

In [ ]:
coefs = log_result.params.drop(labels=['const'])
stdv = np.std(X_train, 0)
feature_importance = abs(coefs * stdv).sort_values(ascending=False)
print(feature_importance)

This table can be interpreted as an approximation of risk factors from high to low according to our injury prediction model. It's a model-specific feature importance method and a global one (as it was gathered from a group of samples). It tells us how much each feature contributes to the variation in injury risk predictions.

For example, if 'Minutes_Played' is at the top of this list, it suggests that differences in playing time have the largest impact on predicted injury risk across our dataset.

## 8. Summary

1. Explainable AI (XAI) in sports analytics provides clear, understandable explanations for predictions, helping coaches, players, and medical staff develop trust in machine learning outputs and make informed decisions.

2. Kernel SHAP is an XAI method that attributes the impact of each input feature (e.g., age, playing time, sprint distance) on a model's prediction of injury risk, providing insights into how these factors influence the model's outcomes.

3. Logistic regression is used here for binary classification (high vs. low injury risk), estimating the probability of injury risk by fitting a linear function to input features and applying a logistic transformation.

4. XAI enhances transparency and accountability in sports analytics, especially crucial when AI-driven decisions can impact athletes' careers and team strategies. It helps demystify complex models, making their reasoning accessible to sports professionals.

5. Machine Learning can assist in predicting injury risks by analyzing various player data, potentially improving player longevity and team performance by enabling preemptive interventions and personalized training regimens.

## Bonus: Serving our Models

To serve models and explainers together, you can run a server with both models using `mlserver`. 
To do so, run the following command.

```sh
python servers/diabetes/cls_diabetes_service.py
```

You can test that your server is working with the following commands.

In [ ]:
from mlserver.codecs import NumpyCodec
import requests

In [ ]:
x.values, y

In [ ]:
inf_request = {
    'inputs': [
        NumpyCodec.encode_input(name='payload', payload=x.values).dict()
    ]
}
print(inf_request)

Change the name from **classifier** to **explainer** and back to change the endpoint you 
are hitting.

In [ ]:
model = 'diabetes_classifier'
endpoint = f"http://0.0.0.0:8080/v2/models/{model}/infer"
r = requests.post(endpoint, json=inf_request)
r.json()

## 10. Exercises

### Build an Explainer on Player Performance Data

- Load a dataset on player performance. You can use statistics from any sport (e.g., basketball, football, tennis). Include about 10 features such as age, games played, key performance indicators, etc.
- Train a logistic regression model to predict high/low performance (you can define this based on a certain threshold).
- Create a Kernel Shap explainer using Alibi, or use another method like AnchorTabular or CounterFactual.
- Write a short narrative describing the results. How could a coach or player use these insights?
- If you choose multiple explainer methods, compare their outputs. Do they highlight similar features? How do their explanations differ?
- Discuss how these explanations could be used in player development, team selection, or game strategy.